# Project 5 — Transformer From Scratch

## Overview
We implement the **Transformer architecture** from scratch using PyTorch — the architecture behind GPT, BERT, and virtually all modern language models. We build a character-level language model on a Shakespeare excerpt.

**Components we build:**
- Multi-Head Self-Attention
- Sinusoidal Positional Encoding
- Feed-Forward Network
- Layer Normalisation
- Residual Connections
- Transformer Block (encoder-style)

## 1. Theory — Attention Mechanism

### Scaled Dot-Product Attention
Given queries $\mathbf{Q}$, keys $\mathbf{K}$, values $\mathbf{V}$:
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

- $\mathbf{Q}, \mathbf{K}$ have shape $(T, d_k)$, $\mathbf{V}$ has shape $(T, d_v)$
- $\frac{1}{\sqrt{d_k}}$ scaling prevents softmax saturation
- The output is a **weighted average of values**, where weights measure query-key similarity

### Multi-Head Attention
Run $h$ attention heads in parallel with different learned projections:
$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}^O$$
$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q, \mathbf{K}\mathbf{W}_i^K, \mathbf{V}\mathbf{W}_i^V)$$

Different heads can attend to different aspects of the input simultaneously.

### Positional Encoding
Transformers have no recurrence so we inject position information via **sinusoidal encoding**:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

These are fixed (not learned) and allow the model to generalise to longer sequences.

### Feed-Forward Sub-Layer
Applied independently to each position:
$$\text{FFN}(x) = \max(0, x\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

The inner dimension is typically $4 \times d_{model}$.

### Layer Norm + Residual Connection
Each sub-layer uses a residual connection and layer normalisation (Post-LN):
$$x_{out} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Dataset — Shakespeare Text

In [ ]:
# Short Shakespeare excerpt (embedded directly — no download needed)
TEXT = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.

Second Citizen:
Would you proceed especially against Caius Marcius?

All:
Against him first: he's a very dog to the commonalty.

Second Citizen:
Consider you what services he has done for his country?

First Citizen:
Very well; and could be content to give him good report fort,
but that he pays himself with being proud.

Second Citizen:
Nay, but speak not maliciously.

First Citizen:
I say unto you, what he hath done famously, he did it to that end:
though soft-conscienced men can be content to say it was for his country
he did it to please his mother and to be partly proud; which he is,
even till the altitude of his virtue.
"""

# Repeat text to get enough training data
TEXT = TEXT * 10

# Build character-level vocabulary
chars     = sorted(set(TEXT))
vocab_size = len(chars)
char2idx  = {c: i for i, c in enumerate(chars)}
idx2char  = {i: c for c, i in char2idx.items()}

# Encode text
data = torch.tensor([char2idx[c] for c in TEXT], dtype=torch.long)

print(f'Text length:  {len(TEXT):,} characters')
print(f'Vocab size:   {vocab_size}')
print(f'Characters:   {"".join(chars[:30])}...')

# Train/val split
n      = len(data)
n_train = int(0.9 * n)
train_data = data[:n_train]
val_data   = data[n_train:]
print(f'Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}')

In [ ]:
# Data batching utility
BLOCK_SIZE = 64   # context length (sequence length)
BATCH_SIZE = 32

def get_batch(split):
    """Sample a random batch of (input, target) sequences."""
    source = train_data if split == 'train' else val_data
    # Random starting positions
    ix = torch.randint(len(source) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([source[i  : i+BLOCK_SIZE  ] for i in ix])  # input tokens
    y  = torch.stack([source[i+1: i+BLOCK_SIZE+1] for i in ix])  # target = shifted by 1
    return x.to(device), y.to(device)

# Test the batch function
xb, yb = get_batch('train')
print(f'Input shape:  {xb.shape}  # (batch, seq_len)')
print(f'Target shape: {yb.shape}')

## 3. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build (max_len, d_model) positional encoding table
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()  # (max_len, 1)
        # Compute division terms: 1 / 10000^(2i/d_model)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)  # even indices
        pe[:, 1::2] = torch.cos(pos * div)  # odd indices

        # Register as buffer (not a parameter, but saved with model)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model) — add position encoding
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Visualise positional encoding
pe_module = PositionalEncoding(d_model=128, max_len=100)
pe_matrix = pe_module.pe.squeeze(0).numpy()  # (100, 128)

plt.figure(figsize=(12, 4))
plt.imshow(pe_matrix[:, :64].T, aspect='auto', cmap='RdBu', origin='lower')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding (first 64 dims)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'

        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads  # dimension per head

        # Separate linear projections for Q, K, V, and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """Reshape (B, T, d_model) → (B, n_heads, T, d_k)."""
        B, T, _ = x.size()
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.size()

        # Project to Q, K, V and split into heads
        Q = self.split_heads(self.W_q(x))  # (B, h, T, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, h, T, T)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)   # (B, h, T, T)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V                      # (B, h, T, d_k)

        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)  # (B, T, d_model)
        return self.W_o(out), attn_weights


print('MultiHeadAttention defined.')

## 5. Transformer Block & Full Model

In [ ]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),          # smoother alternative to ReLU
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """
    One Transformer encoder block:
        x → LayerNorm → MultiHeadAttention → Add
          → LayerNorm → FeedForward       → Add
    Using Pre-LN (layer norm before sublayer) for training stability.
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        # Self-attention sub-layer with residual
        attn_out, _ = self.attn(self.ln1(x), mask)
        x = x + attn_out
        # Feed-forward sub-layer with residual
        x = x + self.ff(self.ln2(x))
        return x


class TransformerLM(nn.Module):
    """Full Transformer Language Model for next-character prediction."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=3,
                 d_ff=512, max_len=512, dropout=0.1):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.blocks  = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_out  = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)

        # Tie input embedding and output weights (reduces parameters)
        self.head.weight = self.embed.weight

    def forward(self, x, targets=None):
        B, T = x.size()

        # Token embeddings + positional encoding
        tok_emb = self.embed(x)     # (B, T, d_model)
        x       = self.pos_enc(tok_emb)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)

        x      = self.ln_out(x)
        logits = self.head(x)  # (B, T, vocab_size)

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss

        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0):
        """Autoregressive generation."""
        self.eval()
        for _ in range(max_new_tokens):
            # Crop context to BLOCK_SIZE
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)
            # Take last position, apply temperature
            logits = logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)
            # Sample next token
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx


# Instantiate
model = TransformerLM(
    vocab_size=vocab_size,
    d_model=128, n_heads=4, n_layers=3,
    d_ff=512, max_len=512
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

## 6. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
TRAIN_STEPS = 500
EVAL_EVERY  = 100
history     = {'train': [], 'val': []}


@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        total = 0
        for _ in range(20):  # average over 20 batches
            xb, yb  = get_batch(split)
            _, loss = model(xb, yb)
            total  += loss.item()
        losses[split] = total / 20
    model.train()
    return losses


model.train()
print(f'Training for {TRAIN_STEPS} steps...')
for step in range(1, TRAIN_STEPS + 1):
    xb, yb   = get_batch('train')
    _, loss  = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % EVAL_EVERY == 0:
        losses = estimate_loss()
        history['train'].append(losses['train'])
        history['val'].append(losses['val'])
        print(f'Step {step:>4}  train loss: {losses["train"]:.4f}  val loss: {losses["val"]:.4f}')

print('Training complete.')

In [ ]:
# Plot loss curves
steps = list(range(EVAL_EVERY, TRAIN_STEPS + 1, EVAL_EVERY))
plt.figure(figsize=(8, 4))
plt.plot(steps, history['train'], label='Train loss')
plt.plot(steps, history['val'],   label='Val loss')
plt.xlabel('Training steps')
plt.ylabel('Cross-entropy loss')
plt.title('Transformer Training Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Generate sample text
print('\n=== Generated Text ===')
seed_text = 'First Citizen:'
seed_idx  = torch.tensor([[char2idx[c] for c in seed_text]], dtype=torch.long).to(device)

generated = model.generate(seed_idx, max_new_tokens=200, temperature=0.8)
generated_text = ''.join([idx2char[i.item()] for i in generated[0]])
print(generated_text)

## Summary

| Component | Role |
|---|---|
| Token embedding | Maps token IDs to vectors |
| Positional encoding | Injects position information |
| Multi-head attention | Computes context-aware representations |
| Feed-forward | Position-wise non-linear transformation |
| Layer norm + residual | Training stability |
| Output projection | Maps to vocabulary logits |

**Key insight**: The attention mechanism allows every token to directly attend to every other token in the sequence — giving $O(1)$ path length between any two positions (vs $O(n)$ for RNNs). This is why transformers handle long-range dependencies so effectively.